In [218]:
import pandas as pd

# ---- Claims ----
df_claims = {
    "fever":       pd.read_csv("_claim_datasets/fever_1k.csv"),
    "fever-fixed": pd.read_csv("_claim_datasets/fever_1k_fixed.csv"),
    "scifact":     pd.read_csv("_claim_datasets/scifact_693.csv"),
    "averitec":    pd.read_csv("_claim_datasets/averitec_3017.csv"),
}

# ---- Results ----
def read_jsonl(path):
    return pd.read_json(path, lines=True)

df_results = {
    "fever": {
        "gpt-4o":       read_jsonl("_results/fever_1k/gpt_results_raw_gpt-4o.jsonl"),
        "gpt-5":        read_jsonl("_results/fever_1k/gpt_results_raw_gpt-5.jsonl"),
        "originality":  read_jsonl("_results/fever_1k/checker_results.jsonl"),
    },
    "scifact": {
        "gpt-4o":       read_jsonl("_results/scifact_693/gpt_results_raw_gpt-4o.jsonl"),
        "gpt-5":        read_jsonl("_results/scifact_693/gpt_results_raw_gpt-5.jsonl"),
        "originality":  read_jsonl("_results/scifact_693/scifact_checker_results.jsonl"),
    },
    "averitec": {
        "gpt-4o":       read_jsonl("_results/averitec_3017/gpt_results_raw_gpt-4o.jsonl"),
        "gpt-5":        read_jsonl("_results/averitec_3017/gpt_results_raw_gpt-5.jsonl"),
        "originality":  read_jsonl("_results/averitec_3017/averitec_checker_results.jsonl"),
    },
}

for dataset in ["fever", "scifact", "averitec"]:
    for model in ["originality", "gpt-4o", "gpt-5"]:
        jsonl = df_results[dataset][model]
        jsonl["id"] = jsonl[f"{dataset}_id"]
        if model == "originality":
            jsonl["explanation"] = jsonl["checker_response"].apply(
                lambda x: x["data"]["results"][0]["explanation"] if x["data"]["results"] else '-'
            )
            jsonl["label"] = jsonl["checker_response"].apply(
                lambda x: x["data"]["results"][0]["classification"] == 'True' if x["data"]["results"] else '-'
            )
        else:
            jsonl["explanation"] = jsonl["response_text"]
            # "The claim is FALSE"
            # "The claim is TRUE"
            jsonl["label"] = jsonl["response_text"].apply(
                lambda x: (True if (x.startswith('TRUE') or x.startswith('The claim is TRUE')) else (False if (x.startswith('FALSE') or x.startswith('The claim is FALSE')) else '-')) if isinstance(x, str) else '-'
            )
        df = jsonl[["id", "claim", "label", "explanation"]]
        df_results[dataset][model] = df


In [219]:
for dataset in ["fever", "fever-fixed", "scifact", "averitec"]:
    print(dataset, df_claims[dataset]["classification"].value_counts(), '\n')

fever classification
False    500
True     500
Name: count, dtype: int64 

fever-fixed classification
False    477
True     450
Name: count, dtype: int64 

scifact classification
True     456
False    237
Name: count, dtype: int64 

averitec classification
False    2047
True      970
Name: count, dtype: int64 



In [220]:
for dataset in ["fever", "fever-fixed", "scifact", "averitec"]:
    rows = []

    for model in ["originality", "gpt-4o", "gpt-5"]:
        
        df_in = df_claims[dataset]
        df_out = df_results["fever" if dataset.startswith("fever") else dataset][model]
        
        id_label = "fever_id" if dataset.startswith("fever") else f"{dataset}_id"
        gold = dict(zip(df_in[id_label], df_in["classification"]))

        df_out_labeled = df_out[(df_out["label"].isin([True, False])) & (df_out["id"].isin(gold.keys()))]

        TP = df_out_labeled.apply(lambda r: gold.get(r["id"]) == True  and r["label"] == True,  axis=1).sum()
        TN = df_out_labeled.apply(lambda r: gold.get(r["id"]) == False and r["label"] == False, axis=1).sum()
        FP = df_out_labeled.apply(lambda r: gold.get(r["id"]) == False and r["label"] == True,  axis=1).sum()
        FN = df_out_labeled.apply(lambda r: gold.get(r["id"]) == True  and r["label"] == False, axis=1).sum()

        assert(TP + TN + FP + FN == len(df_out_labeled))

        classified = len(df_out_labeled)
        unclassified = len(df_in) - classified # don't use len(df_out) for total, since fever_fixed is a reduced set of claims

        accuracy  = (TP + TN) / classified if classified else 0
        precision = TP / (TP + FP) if (TP + FP) else 0
        recall    = TP / (TP + FN) if (TP + FN) else 0
        f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0

        rows.append({
            "model": model,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "TP": f'{TP} ({TP / classified:.3f}%)',
            "TN": f'{TN} ({TN / classified:.3f}%)',
            "FP": f'{FP} ({FP / classified:.3f}%)',
            "FN": f'{FN} ({FN / classified:.3f}%)',
            "unclass.": unclassified,
            "total": classified + unclassified
        })

    df_table = pd.DataFrame(rows)
    print(dataset)
    display(df_table.round(3))
    print()

fever


,model,Accuracy,Precision,Recall,F1,TP,TN,FP,FN,unclass.,total
0,originality,0.904,0.918,0.890,0.904,438 (0.451%),441 (0.454%),39 (0.040%),54 (0.056%),28,1000
1,gpt-4o,0.874,0.868,0.882,0.875,440 (0.440%),433 (0.433%),67 (0.067%),59 (0.059%),1,1000
2,gpt-5,0.916,0.937,0.892,0.914,446 (0.446%),470 (0.470%),30 (0.030%),54 (0.054%),0,1000



fever-fixed


,model,Accuracy,Precision,Recall,F1,TP,TN,FP,FN,unclass.,total
0,originality,0.978,0.980,0.975,0.977,433 (0.478%),452 (0.499%),9 (0.010%),11 (0.012%),22,927
1,gpt-4o,0.941,0.923,0.958,0.940,430 (0.464%),441 (0.476%),36 (0.039%),19 (0.021%),1,927
2,gpt-5,0.996,0.996,0.996,0.996,448 (0.483%),475 (0.512%),2 (0.002%),2 (0.002%),0,927



scifact


,model,Accuracy,Precision,Recall,F1,TP,TN,FP,FN,unclass.,total
0,originality,0.887,0.942,0.884,0.912,403 (0.583%),210 (0.304%),25 (0.036%),53 (0.077%),2,693
1,gpt-4o,0.795,0.845,0.843,0.844,382 (0.554%),166 (0.241%),70 (0.102%),71 (0.103%),4,693
2,gpt-5,0.808,0.938,0.759,0.839,346 (0.499%),214 (0.309%),23 (0.033%),110 (0.159%),0,693



averitec


,model,Accuracy,Precision,Recall,F1,TP,TN,FP,FN,unclass.,total
0,originality,0.827,0.732,0.746,0.739,702 (0.245%),1670 (0.582%),257 (0.090%),239 (0.083%),149,3017
1,gpt-4o,0.809,0.677,0.749,0.711,679 (0.235%),1660 (0.574%),324 (0.112%),228 (0.079%),126,3017
2,gpt-5,0.840,0.801,0.669,0.729,644 (0.215%),1873 (0.625%),160 (0.053%),318 (0.106%),22,3017


In [221]:
rows = []

for model in ["originality", "gpt-4o", "gpt-5"]:
    TP = TN = FP = FN = 0
    classified = unclassified = total = 0

    for dataset in ["fever-fixed", "scifact", "averitec"]:
        df_in = df_claims[dataset]
        df_out = df_results["fever" if dataset.startswith("fever") else dataset][model]

        id_label = "fever_id" if dataset.startswith("fever") else f"{dataset}_id"
        gold = dict(zip(df_in[id_label], df_in["classification"]))

        df_out_labeled = df_out[(df_out["label"].isin([True, False])) & (df_out["id"].isin(gold.keys()))]

        TP += df_out_labeled.apply(lambda r: gold.get(r["id"]) == True  and r["label"] == True,  axis=1).sum()
        TN += df_out_labeled.apply(lambda r: gold.get(r["id"]) == False and r["label"] == False, axis=1).sum()
        FP += df_out_labeled.apply(lambda r: gold.get(r["id"]) == False and r["label"] == True,  axis=1).sum()
        FN += df_out_labeled.apply(lambda r: gold.get(r["id"]) == True  and r["label"] == False, axis=1).sum()

        classified += len(df_out_labeled)
        unclassified += len(df_in) - len(df_out_labeled)
        total += len(df_in)

    accuracy  = (TP + TN) / classified if classified else 0
    precision = TP / (TP + FP) if (TP + FP) else 0
    recall    = TP / (TP + FN) if (TP + FN) else 0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0

    rows.append({
        "model": model,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TP": f'{TP} ({TP / classified:.3f}%)',
        "TN": f'{TN} ({TN / classified:.3f}%)',
        "FP": f'{FP} ({FP / classified:.3f}%)',
        "FN": f'{FN} ({FN / classified:.3f}%)',
        "unclass.": unclassified,
        "total": total
    })

df_combined = pd.DataFrame(rows)
print("Combined (fever-fixed + scifact + averitec)")
display(df_combined.round(4))


Combined (fever-fixed + scifact + averitec)


,model,Accuracy,Precision,Recall,F1,TP,TN,FP,FN,unclass.,total
0,originality,0.8669,0.8409,0.8354,0.8381,1538 (0.345%),2332 (0.522%),291 (0.065%),303 (0.068%),173,4637
1,gpt-4o,0.8340,0.7762,0.8242,0.7995,1491 (0.331%),2267 (0.503%),430 (0.095%),318 (0.071%),131,4637
2,gpt-5,0.8667,0.8860,0.7698,0.8238,1438 (0.312%),2562 (0.555%),185 (0.040%),430 (0.093%),22,4637
